# MR1b — Hourly Crash MapReduce
**Input** : `mr1/crash_zones_mr_ready.csv`  
**Output** : `mr3/hourly_crash_mr.csv`

Schema: `zone_id, hour, avg_crashes, avg_injured, avg_killed, avg_lat, avg_lon`

Groups by `(zone_id, crash_hour)` — computes true hourly average (total ÷ unique dates).

In [10]:
import pandas as pd
import numpy as np
from collections import defaultdict

INPUT  = "crash_zones_mr_ready.csv"
OUTPUT = "hourly_crash_mr.csv"

df = pd.read_csv(INPUT)
df["crash_count"]  = pd.to_numeric(df["crash_count"],  errors="coerce").fillna(0)
df["injury_count"] = pd.to_numeric(df["injury_count"], errors="coerce").fillna(0)
df["death_count"]  = pd.to_numeric(df["death_count"],  errors="coerce").fillna(0)

print(f"Input rows: {len(df):,}")
df.head(10)

Input rows: 2,893


,zone_id,crash_date,crash_hour,crash_count,injury_count,death_count
0,8a2a1008800ffff,2017-02-11,1,1,2.0,0.0
1,8a2a1008800ffff,2019-02-15,20,1,1.0,0.0
2,8a2a1008802ffff,2020-11-05,0,1,0.0,0.0
3,8a2a1008802ffff,2022-10-28,23,1,1.0,0.0
4,8a2a10088047fff,2015-04-01,17,1,0.0,0.0
5,8a2a10088047fff,2020-10-06,20,1,0.0,0.0
6,8a2a10088047fff,2023-10-21,3,1,0.0,0.0
7,8a2a1008804ffff,2015-11-14,6,1,0.0,0.0
8,8a2a1008804ffff,2018-12-04,21,1,1.0,0.0
9,8a2a100880e7fff,2013-11-25,18,1,0.0,0.0


In [8]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# key=(zone_id, crash_hour), val=(crash_count, injury_count, death_count, crash_date)
# avg_lat/avg_lon removed — hex boundaries rendered via zone_id directly.

def mapper(row):
    key = (str(row["zone_id"]), int(row["crash_hour"]))
    val = {
        "crashes": row["crash_count"],
        "injured": row["injury_count"],
        "killed":  row["death_count"],
        "date":    str(row.get("crash_date", "")),
    }
    return key, val

mapped = [mapper(row) for _, row in df.iterrows()]
print(f"Mapped pairs: {len(mapped):,}")

Mapped pairs: 2,893


In [9]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Per (zone_id, crash_hour): sum totals, track unique dates → true hourly avg

acc = defaultdict(lambda: {"crashes":0, "injured":0.0, "killed":0.0, "dates":set()})

for key, val in mapped:
    a = acc[key]
    a["crashes"] += val["crashes"]
    a["injured"] += val["injured"]
    a["killed"]  += val["killed"]
    a["dates"].add(val["date"])

rows = []
for (zone_id, hour), a in acc.items():
    n = max(len(a["dates"]), 1)
    rows.append({
        "zone_id":     zone_id,
        "hour":        hour,
        "avg_crashes": round(a["crashes"] / n, 4),
        "avg_injured": round(a["injured"] / n, 4),
        "avg_killed":  round(a["killed"]  / n, 4),
    })

out = pd.DataFrame(rows).sort_values(["zone_id","hour"]).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f"Output rows   : {len(out):,}")
print(f"Unique zones  : {out['zone_id'].nunique():,}")
print(f"Hours covered : {sorted(out['hour'].unique())}")
print(f"Saved → {OUTPUT}")
out.head(10)

Output rows   : 2,708
Unique zones  : 1,490
Hours covered : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Saved → hourly_crash_mr.csv


,zone_id,hour,avg_crashes,avg_injured,avg_killed
0,8a2a1008800ffff,1,1.0,2.0,0.0
1,8a2a1008800ffff,20,1.0,1.0,0.0
2,8a2a1008802ffff,0,1.0,0.0,0.0
3,8a2a1008802ffff,23,1.0,1.0,0.0
4,8a2a10088047fff,3,1.0,0.0,0.0
5,8a2a10088047fff,17,1.0,0.0,0.0
6,8a2a10088047fff,20,1.0,0.0,0.0
7,8a2a1008804ffff,6,1.0,0.0,0.0
8,8a2a1008804ffff,21,1.0,1.0,0.0
9,8a2a100880e7fff,14,1.0,0.0,0.0
